# 230968078 - Ishan Suryawanshi - Lab8

### Smart Lighting Agent

In [3]:
class SmartLightingAgent:
    def __init__(self):
        self.kb = {
            'MotionDetected': False,
            'IsDark': False,
            'ManualOverride': False,
        }
        self.light_state = 'OFF'
        self.manual_state = 'OFF'

    def perceive(self, motion: bool, is_dark: bool,
                 manual_override: bool, manual_light: str = 'OFF'):
        """Update knowledge base from sensor readings."""
        self.kb['MotionDetected']  = motion
        self.kb['IsDark']          = is_dark
        self.kb['ManualOverride']  = manual_override
        self.manual_state          = manual_light

    def infer(self) -> str:
        """Apply rules to the current KB and return the inferred action."""
        motion   = self.kb['MotionDetected']
        is_dark  = self.kb['IsDark']
        override = self.kb['ManualOverride']

        if override:
            action = f'ManualOverride active → Light stays {self.manual_state}'
            self.light_state = self.manual_state
            return action

        if is_dark and motion:
            self.light_state = 'ON'
            return 'R1 fired (IsDark ∧ MotionDetected) → Light ON'
        if not motion:
            self.light_state = 'OFF'
            return 'R2 fired (¬MotionDetected) → Light OFF'


        return 'No rule matched → Light stays ' + self.light_state

    def run(self, percepts: dict) -> str:
        self.perceive(**percepts)
        return self.infer()


In [2]:
agent = SmartLightingAgent()

test_cases = [
    {'motion': True,  'is_dark': True,  'manual_override': False, 'manual_light': 'OFF'},
    {'motion': False, 'is_dark': True,  'manual_override': False, 'manual_light': 'OFF'},
    {'motion': True,  'is_dark': False, 'manual_override': False, 'manual_light': 'OFF'},
    {'motion': True,  'is_dark': True,  'manual_override': True,  'manual_light': 'OFF'},
    {'motion': False, 'is_dark': False, 'manual_override': True,  'manual_light': 'ON'},
]

print('=== Smart Lighting Agent ===')
print(f'{"Percepts":<55} {"Action"}')
print('-' * 90)
for p in test_cases:
    action = agent.run(p)
    percept_str = (f'Motion={p["motion"]}, Dark={p["is_dark"]}, '
                   f'Override={p["manual_override"]}')
    print(f'{percept_str:<55} {action}')

=== Smart Lighting Agent ===
Percepts                                                Action
------------------------------------------------------------------------------------------
Motion=True, Dark=True, Override=False                  R1 fired (IsDark ∧ MotionDetected) → Light ON
Motion=False, Dark=True, Override=False                 R2 fired (¬MotionDetected) → Light OFF
Motion=True, Dark=False, Override=False                 No rule matched → Light stays OFF
Motion=True, Dark=True, Override=True                   ManualOverride active → Light stays OFF
Motion=False, Dark=False, Override=True                 ManualOverride active → Light stays ON


### Medical Diagnosis Logical Agent

In [4]:
class MedicalAgent:
    def __init__(self):
        self.rules = [
            (['Fever', 'Cough'],                        'Flu'),
            (['Fever', 'Cough', 'ChestPain'],           'Pneumonia'),
            (['Pneumonia'],                             'HospitalizationRequired'),
            (['Flu'],                                   'RestAndFluids'),
        ]
        self.facts = set()

    def set_symptoms(self, **symptoms):
        self.facts = {name for name, present in symptoms.items() if present}
        print(f'Observed symptoms : {self.facts if self.facts else "None"}')

    def forward_chain(self):
        trace = []
        changed = True
        while changed:
            changed = False
            for premises, conclusion in self.rules:
                if all(p in self.facts for p in premises) and conclusion not in self.facts:
                    self.facts.add(conclusion)
                    trace.append((premises, conclusion))
                    changed = True
        return trace

    def diagnose(self, **symptoms):
        self.set_symptoms(**symptoms)
        trace = self.forward_chain()

        print('\nForward-chaining trace:')
        if trace:
            for step, (premises, conclusion) in enumerate(trace, 1):
                print(f'  Step {step}: {" ∧ ".join(premises)} → {conclusion}')
        else:
            print('  No rules fired.')

        diseases = [f for f in self.facts if f in ('Flu', 'Pneumonia')]
        actions  = [f for f in self.facts if f in ('HospitalizationRequired', 'RestAndFluids')]

        print('\n── Diagnosis ──')
        print(f'  Possible diseases : {diseases if diseases else ["Unknown"]}')
        print(f'  Required actions  : {actions  if actions  else ["Monitor and reassess"]}')
        print()

In [5]:
print('=== Medical Diagnosis Agent ===')

print('\n--- Case A: Fever + Cough ---')
agent2a = MedicalAgent()
agent2a.diagnose(Fever=True, Cough=True, ChestPain=False)

print('--- Case B: Fever + Cough + ChestPain ---')
agent2b = MedicalAgent()
agent2b.diagnose(Fever=True, Cough=True, ChestPain=True)

print('--- Case C: No symptoms ---')
agent2c = MedicalAgent()
agent2c.diagnose(Fever=False, Cough=False, ChestPain=False)

=== Medical Diagnosis Agent ===

--- Case A: Fever + Cough ---
Observed symptoms : {'Fever', 'Cough'}

Forward-chaining trace:
  Step 1: Fever ∧ Cough → Flu
  Step 2: Flu → RestAndFluids

── Diagnosis ──
  Possible diseases : ['Flu']
  Required actions  : ['RestAndFluids']

--- Case B: Fever + Cough + ChestPain ---
Observed symptoms : {'ChestPain', 'Fever', 'Cough'}

Forward-chaining trace:
  Step 1: Fever ∧ Cough → Flu
  Step 2: Fever ∧ Cough ∧ ChestPain → Pneumonia
  Step 3: Pneumonia → HospitalizationRequired
  Step 4: Flu → RestAndFluids

── Diagnosis ──
  Possible diseases : ['Flu', 'Pneumonia']
  Required actions  : ['HospitalizationRequired', 'RestAndFluids']

--- Case C: No symptoms ---
Observed symptoms : None

Forward-chaining trace:
  No rules fired.

── Diagnosis ──
  Possible diseases : ['Unknown']
  Required actions  : ['Monitor and reassess']



### Traffic Violation Detection Logical Agent

In [6]:
class TrafficAgent:
    def __init__(self):
        self.kb = {}
        self.violations = []
        self.actions    = []

    def perceive(self, speed_above: bool, signal_red: bool,
                 crossed_line: bool, emergency: bool):
        self.kb = {
            'SpeedAboveLimit':    speed_above,
            'SignalRed':          signal_red,
            'VehicleCrossedLine': crossed_line,
            'EmergencyVehicle':   emergency,
        }
        self.violations = []
        self.actions    = []

    def infer(self):
        kb = self.kb
        emergency = kb['EmergencyVehicle']

        if emergency:
            self.kb['ExemptFromRules'] = True
            self.actions.append('ExemptFromRules — Emergency vehicle, no enforcement')
            return

        if kb['SpeedAboveLimit']:
            self.violations.append('SpeedViolation')
            self.actions.append('Issue Speed Ticket')

        if kb['SignalRed'] and kb['VehicleCrossedLine']:
            self.violations.append('SignalViolation')
            self.actions.append('Issue Signal Ticket')

        if not self.violations:
            self.actions.append('No violations detected — No action')

    def run(self, **percepts):
        self.perceive(**percepts)
        self.infer()
        return self.violations, self.actions

In [7]:
agent3 = TrafficAgent()

timeline = [
    {'speed_above': False, 'signal_red': False, 'crossed_line': False, 'emergency': False},
    {'speed_above': True,  'signal_red': False, 'crossed_line': False, 'emergency': False},
    {'speed_above': False, 'signal_red': True,  'crossed_line': True,  'emergency': False},
    {'speed_above': True,  'signal_red': True,  'crossed_line': True,  'emergency': False},
    {'speed_above': True,  'signal_red': True,  'crossed_line': True,  'emergency': True},
]

print('=== Traffic Violation Detection Agent ===')
for t, p in enumerate(timeline, 1):
    violations, actions = agent3.run(**p)
    print(f'\nTime step {t}: Speed={p["speed_above"]}, RedLight={p["signal_red"]}, '
          f'Crossed={p["crossed_line"]}, Emergency={p["emergency"]}')
    print(f'  Violations : {violations if violations else "None"}')
    print(f'  Actions    : {actions}')

=== Traffic Violation Detection Agent ===

Time step 1: Speed=False, RedLight=False, Crossed=False, Emergency=False
  Violations : None
  Actions    : ['No violations detected — No action']

Time step 2: Speed=True, RedLight=False, Crossed=False, Emergency=False
  Violations : ['SpeedViolation']
  Actions    : ['Issue Speed Ticket']

Time step 3: Speed=False, RedLight=True, Crossed=True, Emergency=False
  Violations : ['SignalViolation']
  Actions    : ['Issue Signal Ticket']

Time step 4: Speed=True, RedLight=True, Crossed=True, Emergency=False
  Violations : ['SpeedViolation', 'SignalViolation']
  Actions    : ['Issue Speed Ticket', 'Issue Signal Ticket']

Time step 5: Speed=True, RedLight=True, Crossed=True, Emergency=True
  Violations : None
  Actions    : ['ExemptFromRules — Emergency vehicle, no enforcement']


### Cybersecurity Intrusion Detection Logical Agent

In [8]:
class CyberSecurityAgent:

    THREAT_LEVELS = {
        'SuspiciousLogin':    'MEDIUM',
        'DataBreachRisk':     'HIGH',
        'HighRiskIntrusion':  'CRITICAL',
    }

    def __init__(self):
        self.rules = [
            (['MultipleFailedLogins', 'LoginFromUnknownIP'],     'SuspiciousLogin'),
            (['SuspiciousLogin',      'AdminPrivileges'],         'HighRiskIntrusion'),
            (['AccessToSensitiveFiles', 'SuspiciousLogin'],       'DataBreachRisk'),
            (['HighRiskIntrusion'],                               'TriggerAlert'),
            (['HighRiskIntrusion'],                               'LockAccount'),
            (['DataBreachRisk'],                                  'NotifySecurityTeam'),
        ]
        self.facts = set()

    def update_facts(self, **events):
        """Add or remove facts based on current sensor events."""
        for fact, present in events.items():
            if present:
                self.facts.add(fact)
            else:
                self.facts.discard(fact)

    def forward_chain(self):
        changed = True
        new_inferences = []
        while changed:
            changed = False
            for premises, conclusion in self.rules:
                if all(p in self.facts for p in premises) and conclusion not in self.facts:
                    self.facts.add(conclusion)
                    new_inferences.append((premises, conclusion))
                    changed = True
        return new_inferences

    def get_threat_level(self):
        for threat, level in self.THREAT_LEVELS.items():
            if threat in self.facts:
                return level
        return 'LOW'

    def get_actions(self):
        return [f for f in ['TriggerAlert', 'LockAccount', 'NotifySecurityTeam']
                if f in self.facts]

    def run(self, **events):
        self.update_facts(**events)
        trace = self.forward_chain()
        return trace, self.get_threat_level(), self.get_actions()

In [9]:
agent4 = CyberSecurityAgent()

event_stream = [
    {'MultipleFailedLogins': True,  'LoginFromUnknownIP': True,
     'AccessToSensitiveFiles': False, 'AdminPrivileges': False},
    {'MultipleFailedLogins': True,  'LoginFromUnknownIP': True,
     'AccessToSensitiveFiles': True,  'AdminPrivileges': False},
    {'MultipleFailedLogins': True,  'LoginFromUnknownIP': True,
     'AccessToSensitiveFiles': True,  'AdminPrivileges': True},
]

print('=== Cybersecurity Intrusion Detection Agent ===')
for step, events in enumerate(event_stream, 1):
    trace, threat, actions = agent4.run(**events)
    print(f'\n── Event stream step {step} ──')
    print(f'  Active events : {[k for k,v in events.items() if v]}')
    if trace:
        print('  New inferences:')
        for premises, conclusion in trace:
            print(f'    {" ∧ ".join(premises)} → {conclusion}')
    print(f'  Threat level  : {threat}')
    print(f'  Actions taken : {actions if actions else ["Monitor"]}')

=== Cybersecurity Intrusion Detection Agent ===

── Event stream step 1 ──
  Active events : ['MultipleFailedLogins', 'LoginFromUnknownIP']
  New inferences:
    MultipleFailedLogins ∧ LoginFromUnknownIP → SuspiciousLogin
  Threat level  : MEDIUM
  Actions taken : ['Monitor']

── Event stream step 2 ──
  Active events : ['MultipleFailedLogins', 'LoginFromUnknownIP', 'AccessToSensitiveFiles']
  New inferences:
    AccessToSensitiveFiles ∧ SuspiciousLogin → DataBreachRisk
    DataBreachRisk → NotifySecurityTeam
  Threat level  : MEDIUM
  Actions taken : ['NotifySecurityTeam']

── Event stream step 3 ──
  Active events : ['MultipleFailedLogins', 'LoginFromUnknownIP', 'AccessToSensitiveFiles', 'AdminPrivileges']
  New inferences:
    SuspiciousLogin ∧ AdminPrivileges → HighRiskIntrusion
    HighRiskIntrusion → TriggerAlert
    HighRiskIntrusion → LockAccount
  Threat level  : MEDIUM
  Actions taken : ['TriggerAlert', 'LockAccount', 'NotifySecurityTeam']


### Autonomous warehouse robot

In [10]:
# ─────────────────────────────────────────────────────────────
# PROBLEM 5 — Autonomous Warehouse Robot
# ─────────────────────────────────────────────────────────────

class WarehouseRobot:
    def __init__(self, robot_id='R1'):
        self.robot_id = robot_id
        self.kb = {
            'ObstacleAhead':      False,
            'PathClear':          True,
            'CarryingFragileItem':False,
            'HumanNearby':        False,
            'DestinationReached': False,
        }

    def infer_action(self) -> tuple:
        """
        Returns (action, rule_fired, explanation).
        Rules are checked in descending priority.
        """
        kb = self.kb

        if kb['DestinationReached']:
            return ('StopAndUnload', 'R5',
                    'DestinationReached → StopAndUnload')

        if kb['CarryingFragileItem'] and kb['ObstacleAhead']:
            return ('SlowDown', 'R2',
                    'CarryingFragileItem ∧ ObstacleAhead → SlowDown')

        if kb['HumanNearby']:
            return ('ReduceSpeed', 'R3',
                    'HumanNearby → ReduceSpeed')

        if kb['ObstacleAhead']:
            return ('Stop', 'R1',
                    'ObstacleAhead ∧ ¬CarryingFragileItem → Stop')

        if kb['PathClear'] and not kb['HumanNearby']:
            return ('MoveForward', 'R4',
                    'PathClear ∧ ¬HumanNearby → MoveForward')

        return ('Idle', 'None', 'No rule matched — waiting')

    def perceive(self, **sensor_readings):
        self.kb.update(sensor_readings)

    def act(self):
        action, rule, explanation = self.infer_action()
        return action, rule, explanation

    def run(self, **sensor_readings):
        self.perceive(**sensor_readings)
        return self.act()

In [11]:
robot = WarehouseRobot('WH-BOT-01')

scenarios = [
    {'ObstacleAhead': False, 'PathClear': True,  'CarryingFragileItem': False,
     'HumanNearby': False, 'DestinationReached': False},

    {'ObstacleAhead': True,  'PathClear': False, 'CarryingFragileItem': False,
     'HumanNearby': False, 'DestinationReached': False},

    {'ObstacleAhead': True,  'PathClear': False, 'CarryingFragileItem': True,
     'HumanNearby': False, 'DestinationReached': False},

    {'ObstacleAhead': False, 'PathClear': True,  'CarryingFragileItem': True,
     'HumanNearby': True,  'DestinationReached': False},

    {'ObstacleAhead': False, 'PathClear': True,  'CarryingFragileItem': True,
     'HumanNearby': False, 'DestinationReached': True},
]

print('=== Autonomous Warehouse Robot ===')
for i, s in enumerate(scenarios, 1):
    action, rule, explanation = robot.run(**s)
    active = [k for k, v in s.items() if v]
    print(f'\nScenario {i}: {active}')
    print(f'  Rule fired : {rule}')
    print(f'  Inference  : {explanation}')
    print(f'  Action     : *** {action} ***')

=== Autonomous Warehouse Robot ===

Scenario 1: ['PathClear']
  Rule fired : R4
  Inference  : PathClear ∧ ¬HumanNearby → MoveForward
  Action     : *** MoveForward ***

Scenario 2: ['ObstacleAhead']
  Rule fired : R1
  Inference  : ObstacleAhead ∧ ¬CarryingFragileItem → Stop
  Action     : *** Stop ***

Scenario 3: ['ObstacleAhead', 'CarryingFragileItem']
  Rule fired : R2
  Inference  : CarryingFragileItem ∧ ObstacleAhead → SlowDown
  Action     : *** SlowDown ***

Scenario 4: ['PathClear', 'CarryingFragileItem', 'HumanNearby']
  Rule fired : R3
  Inference  : HumanNearby → ReduceSpeed
  Action     : *** ReduceSpeed ***

Scenario 5: ['PathClear', 'CarryingFragileItem', 'DestinationReached']
  Rule fired : R5
  Inference  : DestinationReached → StopAndUnload
  Action     : *** StopAndUnload ***
